In [1]:
import cv2
import os,socket,sys,time
import spidev as SPI
import xgoscreen.LCD_2inch as LCD_2inch
from PIL import Image,ImageDraw,ImageFont
from key import Button
import numpy as np

from picamera2 import Picamera2

In [2]:

mydisplay = LCD_2inch.LCD_2inch()
mydisplay.clear()
splash = Image.new("RGB", (mydisplay.height, mydisplay.width ),"black")
mydisplay.ShowImage(splash)
button=Button()

In [3]:
# 导入组件 Importing Components
import ipywidgets.widgets as widgets
image_widget = widgets.Image(format='jpeg', width=320, height=240)  #设置摄像头显示组件  Set up the camera display component

# 将BGR图像转换为JPEG格式的字节流 Convert a BGR image to a JPEG byte stream
def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])

display(image_widget) #显示出来

Image(value=b'', format='jpeg', height='240', width='320')

In [4]:
#-----------------------COMMON INIT-----------------------
import pyzbar.pyzbar as pyzbar

def cv2AddChineseText(img, text, position, textColor=(0, 255, 0), textSize=30):
    if (isinstance(img, np.ndarray)):  
        img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img)
    fontStyle = ImageFont.truetype(
        "/home/pi/model/msyh.ttc", textSize, encoding="utf-8")
    draw.text(position, text, textColor, font=fontStyle)
    return cv2.cvtColor(np.asarray(img), cv2.COLOR_RGB2BGR)

 
picam2 = Picamera2()
picam2.configure(
    picam2.create_preview_configuration(main={"format": "RGB888", "size": (320, 240)})
)
picam2.start()

[0:20:30.225965198] [134792]  INFO Camera camera_manager.cpp:325 libcamera v0.3.2+99-1230f78d
[0:20:30.236130995] [134880]  INFO RPI pisp.cpp:695 libpisp version v1.0.7 28196ed6edcf 29-08-2024 (16:33:32)
[0:20:30.269703042] [134880]  INFO RPI pisp.cpp:1154 Registered camera /base/axi/pcie@120000/rp1/i2c@88000/ov5647@36 to CFE device /dev/media2 and ISP device /dev/media0 using PiSP variant BCM2712_D0
[0:20:30.273582929] [134792]  INFO Camera camera.cpp:1197 configuring streams: (0) 320x240-RGB888 (1) 640x480-GBRG_PISP_COMP1
[0:20:30.273680615] [134880]  INFO RPI pisp.cpp:1450 Sensor: /base/axi/pcie@120000/rp1/i2c@88000/ov5647@36 - Selected sensor format: 640x480-SGBRG10_1X10 - Selected CFE format: 640x480-PC1g


In [5]:
while(True):
    img = picam2.capture_array() 
    frame = cv2.flip(img, 1)
    img_ROI_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    barcodes = pyzbar.decode(img_ROI_gray) 
    for barcode in barcodes:
        barcodeData = barcode.data.decode("utf-8")
        barcodeType = barcode.type
        text = "{} ({})".format(barcodeData, barcodeType)
        img=cv2AddChineseText(img,text, (10, 30),(0, 255, 0), 30)
        print("[INFO] Found {} barcode: {}".format(barcodeType, barcodeData))

    
    b,g,r = cv2.split(img)
    img = cv2.merge((r,g,b))
    imgok = Image.fromarray(img)
    mydisplay.ShowImage(imgok)

    r,g,b = cv2.split(img)
    img1 = cv2.merge((b,g,r))
    image_widget.value = bgr8_to_jpeg(img1)

    if (cv2.waitKey(1)) == ord('q'):
        break
    if button.press_b():
        break

[INFO] Found QRCODE barcode: https://wx.openjia.com/users?mode=2&hpwd=f390fcbda6a92115225820d1
[INFO] Found QRCODE barcode: https://wx.openjia.com/users?mode=2&hpwd=f390fcbda6a92115225820d1
[INFO] Found QRCODE barcode: https://wx.openjia.com/users?mode=2&hpwd=f390fcbda6a92115225820d1
[INFO] Found QRCODE barcode: https://wx.openjia.com/users?mode=2&hpwd=f390fcbda6a92115225820d1
[INFO] Found QRCODE barcode: https://wx.openjia.com/users?mode=2&hpwd=f390fcbda6a92115225820d1
[INFO] Found QRCODE barcode: https://wx.openjia.com/users?mode=2&hpwd=f390fcbda6a92115225820d1
[INFO] Found QRCODE barcode: https://wx.openjia.com/users?mode=2&hpwd=f390fcbda6a92115225820d1
[INFO] Found QRCODE barcode: https://wx.openjia.com/users?mode=2&hpwd=f390fcbda6a92115225820d1
[INFO] Found QRCODE barcode: https://wx.openjia.com/users?mode=2&hpwd=f390fcbda6a92115225820d1
[INFO] Found QRCODE barcode: https://wx.openjia.com/users?mode=2&hpwd=f390fcbda6a92115225820d1
[INFO] Found QRCODE barcode: https://wx.openjia.co

KeyboardInterrupt: 

In [ ]:
del mydisplay
picam2.stop()
picam2.close()